In [ ]:
import os
import xml.etree.ElementTree as ET
# XML 파일들이 있는 경로
input_dir = 'C:/Users/AI-LHJ/Desktop/EGG/Training/label'  # 사용자에 맞게 경로를 변경해주세요
output_dir = 'C:/Users/AI-LHJ/Desktop/EGG/Training/labels'  # YOLO 형식의 라벨을 저장할 폴더
os.makedirs(output_dir, exist_ok=True)
files = [f for f in os.listdir(input_dir) if f.endswith('.xml')]
for file in files:
    # XML 파일 파싱
    tree = ET.parse(os.path.join(input_dir, file))
    root = tree.getroot()

    # 이미지 크기 정보
    img_width = int(root.find('size/width').text)
    img_height = int(root.find('size/height').text)

    # YOLO 형식 데이터로 변환
    yolo_data = []
    for obj in root.findall('bndbox'):
        # 클래스 ID 가져오기 (여기서는 'state' 태그를 클래스 ID로 사용)
        class_id = int(obj.find('state').text) - 1  # YOLO는 0부터 시작하므로 1을 뺌

        # 바운딩 박스 정보
        x_min = float(obj.find('x_min').text)
        y_min = float(obj.find('y_min').text)
        x_max = float(obj.find('x_max').text)
        y_max = float(obj.find('y_max').text)

        # 중심점, 너비, 높이 계산
        x_center = (x_min + x_max) / 2.0 / img_width
        y_center = (y_min + y_max) / 2.0 / img_height
        width = (x_max - x_min) / img_width
        height = (y_max - y_min) / img_height

        # YOLO 형식의 라벨 데이터 추가
        yolo_data.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    # 원본 파일명에서 확장자를 .txt로 변경하여 저장
    output_file = os.path.join(output_dir, os.path.splitext(file)[0] + '.txt')
    with open(output_file, 'w') as f:
        f.write("\n".join(yolo_data))

print(f"YOLO 형식의 라벨 파일들이 {output_dir}에 저장되었습니다.")

In [ ]:
import os
import random
import shutil

# 경로 설정
base_path = r"C:/Users/AI-LHJ/Desktop/EGG/Training"
train_path = os.path.join(base_path, 'train')
valid_path = os.path.join(base_path, 'vaild')
test_path = os.path.join(base_path, 'test')

# 이미지와 라벨 파일 경로
train_images_path = os.path.join(train_path, 'images')
train_labels_path = os.path.join(train_path, 'labels')

valid_images_path = os.path.join(valid_path, 'images')
valid_labels_path = os.path.join(valid_path, 'labels')

test_images_path = os.path.join(test_path, 'images')
test_labels_path = os.path.join(test_path, 'labels')

# 폴더가 없으면 생성
os.makedirs(valid_images_path, exist_ok=True)
os.makedirs(valid_labels_path, exist_ok=True)
os.makedirs(test_images_path, exist_ok=True)
os.makedirs(test_labels_path, exist_ok=True)

# train 폴더에 있는 모든 이미지 파일 리스트
images = os.listdir(train_images_path)

# 파일을 랜덤하게 섞기
random.shuffle(images)

# 데이터 비율 계산 (7:2:1)
total = len(images)
train_split = int(total * 0.7)
valid_split = int(total * 0.9)  # 70% train + 20% valid

# train:valid:test로 나누기
train_images = images[:train_split]
valid_images = images[train_split:valid_split]
test_images = images[valid_split:]

def move_files(file_list, src_images, src_labels, dst_images, dst_labels):
    for image_file in file_list:
        # 이미지 파일 이동
        src_image = os.path.join(src_images, image_file)
        dst_image = os.path.join(dst_images, image_file)
        shutil.move(src_image, dst_image)

        # 라벨 파일은 이미지 파일과 동일한 이름에 확장자만 .txt
        label_file = image_file.replace('.jpg', '.txt')  # 확장자에 맞게 수정 (.jpg -> .txt)
        src_label = os.path.join(src_labels, label_file)
        dst_label = os.path.join(dst_labels, label_file)

        # 라벨 파일이 존재할 경우 이동
        if os.path.exists(src_label):
            shutil.move(src_label, dst_label)
        else:
            print(f"Warning: {label_file} 파일을 찾을 수 없습니다.")

# 파일 이동 (train -> valid, test)
move_files(valid_images, train_images_path, train_labels_path, valid_images_path, valid_labels_path)
move_files(test_images, train_images_path, train_labels_path, test_images_path, test_labels_path)

print("데이터가 7:2:1 비율로 나누어졌습니다.")


In [ ]:
from ultralytics import YOLO
from multiprocessing import freeze_support

# 모델 경로 설정
model_path = 'yolov8m.pt'
# 데이터 설정 파일 경로
data_path = 'C:/Users/AI-LHJ/Desktop/EGG/data.yaml'
# 학습 결과 저장 경로
project_path = 'C:/Users/AI-LHJ/Desktop/EGG/train_result'

# YOLO 모델 불러오기
model = YOLO(model_path)

if __name__ == '__main__':
    freeze_support()
    
    # 모델 학습
    model.train(data=data_path, epochs=100, project=project_path, name='color_egg_m')

In [ ]:
from ultralytics import YOLO

# 모델 로드
model = YOLO("C:/Users/AI-LHJ/Desktop/ultralytics-main/runs/detect/train/weights/best.pt")  # 사용자 정의 모델을 로드합니다

# 모델 검증
metrics = model.val()  # 인자가 필요 없음, 데이터셋과 설정이 기억됩니다
metrics.box.map    # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps   # 각 카테고리의 map50-95가 포함된 목록